# 08 - Gold | Perfil Geracional de Consumo

## Objetivo

Construir o perfil de consumo digital por faixa etária a partir
dos microdados da TIC Domicílios 2025.

A análise considera o peso amostral disponibilizado pelo CETIC.br,
permitindo estimar a proporção ponderada de indivíduos que consomem
cada categoria de conteúdo.

## Granularidade

Uma linha representa:

`faixa_etaria + categoria`

## Categorias

- Notícias
- Esportes
- Música
- Humor
- Games
- Animações
- Tutoriais / Educação
- Influenciadores

## Escopos de comparação

### Núcleo multifuente
Categorias comparáveis entre CETIC, YouTube e Google Trends:

- Notícias
- Esportes
- Música
- Humor
- Games

### Comparação ampliada
Categorias adicionais comparáveis entre CETIC e YouTube:

- Animações
- Tutoriais / Educação
- Influenciadores

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
df_cetic = spark.table(
    "workspace.mvp_silver.consumo_digital_cetic_2025"
)

print("Registros Silver:", df_cetic.count())

In [0]:
display(
    df_cetic
    .select(
        "id_respondente",
        "faixa_etaria_codigo",
        "faixa_etaria",
        "categoria",
        "resposta",
        "resposta_valida",
        "peso"
    )
    .limit(20)
)

In [0]:
df_cetic_validos = (
    df_cetic
    .filter(
        F.col("resposta_valida") == True
    )
)

print(
    "Registros com resposta válida:",
    df_cetic_validos.count()
)

In [0]:
display(
    df_cetic_validos
    .groupBy(
        "resposta",
        "status_resposta"
    )
    .count()
    .orderBy("resposta")
)

In [0]:
df_cetic_validos = (
    df_cetic_validos
    .withColumn(
        "peso_consumo",
        F.col("peso") * F.col("resposta")
    )
)

In [0]:
display(
    df_cetic_validos
    .select(
        "id_respondente",
        "faixa_etaria",
        "categoria",
        "resposta",
        "peso",
        "peso_consumo"
    )
    .limit(30)
)

In [0]:
df_perfil_base = (
    df_cetic_validos
    .groupBy(
        "faixa_etaria_codigo",
        "faixa_etaria",
        "categoria"
    )
    .agg(
        F.count("*").alias(
            "respondentes_validos"
        ),

        F.sum(
            F.when(
                F.col("resposta") == 1,
                1
            ).otherwise(0)
        ).alias(
            "respondentes_sim"
        ),

        F.sum("peso").alias(
            "peso_total_valido"
        ),

        F.sum("peso_consumo").alias(
            "peso_consumo"
        )
    )
)

In [0]:
df_perfil_gold = (
    df_perfil_base
    .withColumn(
        "percentual_consumo",
        F.round(
            (
                F.col("peso_consumo")
                /
                F.col("peso_total_valido")
            ) * 100,
            2
        )
    )
)

In [0]:
df_perfil_gold = (
    df_perfil_gold
    .withColumn(
        "percentual_consumo_nao_ponderado",
        F.round(
            (
                F.col("respondentes_sim")
                /
                F.col("respondentes_validos")
            ) * 100,
            2
        )
    )
)

In [0]:
categorias_nucleo = [
    "Notícias",
    "Esportes",
    "Música",
    "Humor",
    "Games"
]

df_perfil_gold = (
    df_perfil_gold
    .withColumn(
        "escopo_comparacao",
        F.when(
            F.col("categoria").isin(
                categorias_nucleo
            ),
            "Núcleo - CETIC + YouTube + Google Trends"
        )
        .otherwise(
            "Ampliado - CETIC + YouTube"
        )
    )
)

In [0]:
df_perfil_gold = (
    df_perfil_gold
    .withColumn(
        "ordem_faixa_etaria",
        F.col("faixa_etaria_codigo")
    )
)

In [0]:
janela_ranking = (
    Window
    .partitionBy("faixa_etaria")
    .orderBy(
        F.desc("percentual_consumo")
    )
)

df_perfil_gold = (
    df_perfil_gold
    .withColumn(
        "ranking_categoria_na_faixa",
        F.dense_rank().over(
            janela_ranking
        )
    )
)

In [0]:
df_perfil_gold = (
    df_perfil_gold
    .withColumn(
        "ano_referencia",
        F.lit(2025)
    )
    .withColumn(
        "fonte",
        F.lit(
            "CETIC.br - TIC Domicílios 2025"
        )
    )
    .withColumn(
        "_data_processamento",
        F.current_timestamp()
    )
)

In [0]:
df_perfil_gold = (
    df_perfil_gold
    .select(
        "ano_referencia",
        "ordem_faixa_etaria",
        "faixa_etaria",
        "categoria",
        "escopo_comparacao",
        "respondentes_validos",
        "respondentes_sim",
        "peso_total_valido",
        "peso_consumo",
        "percentual_consumo",
        "percentual_consumo_nao_ponderado",
        "ranking_categoria_na_faixa",
        "fonte",
        "_data_processamento"
    )
)

In [0]:
display(
    df_perfil_gold
    .orderBy(
        "ordem_faixa_etaria",
        "ranking_categoria_na_faixa"
    )
)

In [0]:
print(
    "Linhas Gold:",
    df_perfil_gold.count()
)

print(
    "Faixas etárias:",
    df_perfil_gold
    .select("faixa_etaria")
    .distinct()
    .count()
)

print(
    "Categorias:",
    df_perfil_gold
    .select("categoria")
    .distinct()
    .count()
)

In [0]:
duplicados_gold = (
    df_perfil_gold
    .groupBy(
        "faixa_etaria",
        "categoria"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

print(
    "Duplicidades faixa + categoria:",
    duplicados_gold.count()
)

In [0]:
percentuais_invalidos = (
    df_perfil_gold
    .filter(
        (F.col("percentual_consumo") < 0)
        |
        (F.col("percentual_consumo") > 100)
    )
    .count()
)

print(
    "Percentuais fora de 0-100:",
    percentuais_invalidos
)

In [0]:
display(
    df_perfil_gold
    .select(
        "faixa_etaria",
        "categoria",
        "percentual_consumo",
        "ranking_categoria_na_faixa"
    )
    .orderBy(
        "ordem_faixa_etaria",
        "ranking_categoria_na_faixa"
    )
)

In [0]:
display(
    df_perfil_gold
    .filter(
        F.col("ranking_categoria_na_faixa") == 1
    )
    .select(
        "faixa_etaria",
        "categoria",
        "percentual_consumo"
    )
    .orderBy(
        "ordem_faixa_etaria"
    )
)

In [0]:
display(
    df_perfil_gold
    .filter(
        F.col("ranking_categoria_na_faixa") == 1
    )
    .select(
        "faixa_etaria",
        "categoria",
        "percentual_consumo"
    )
    .orderBy(
        "ordem_faixa_etaria"
    )
)

In [0]:
(
    df_perfil_gold.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        "workspace.mvp_gold.perfil_geracional_cetic_2025"
    )
)

In [0]:
df_gold_check = spark.table(
    "workspace.mvp_gold.perfil_geracional_cetic_2025"
)

print(
    "Linhas gravadas:",
    df_gold_check.count()
)

display(
    df_gold_check
    .orderBy(
        "ordem_faixa_etaria",
        "ranking_categoria_na_faixa"
    )
)

In [0]:
%sql

SHOW TABLES IN workspace.mvp_gold;

## Resultado da Gold - Perfil Geracional

A camada Gold consolidou o consumo digital da TIC Domicílios 2025
em uma granularidade de faixa etária e categoria de conteúdo.

Foram produzidos 48 registros analíticos:

`6 faixas etárias × 8 categorias = 48 combinações`

O indicador principal, `percentual_consumo`, foi calculado utilizando
os pesos amostrais disponibilizados pelo CETIC.br.

A fórmula adotada foi:

`Σ peso dos respondentes que responderam Sim /
 Σ peso das respostas válidas × 100`

Também foi preservado o percentual não ponderado para fins de
comparação metodológica.

Cada categoria foi ranqueada dentro da respectiva faixa etária,
permitindo identificar os tipos de conteúdo com maior incidência
de consumo em cada grupo.

A tabela diferencia ainda dois escopos analíticos:

- Núcleo: CETIC + YouTube + Google Trends
- Ampliado: CETIC + YouTube

Tabela criada:

`workspace.mvp_gold.perfil_geracional_cetic_2025`